# 08 — Segment × Attribute Correlation Analysis

Analyses the pre-computed correlation tables from `data/correlations/`.  
Each file maps a `(user_characteristic, value)` segment — e.g. `country=US` or `target_age_segment=35-44` — to a signed correlation between a creative attribute and a performance metric.

Two methods are compared:
- **statistical** — Pearson *r* (with p-value)
- **rf_signed** — Random Forest importance × `sign(Pearson)`, capturing non-linear relationships

Run `uv run python scripts/precompute_correlations.py --all` first to generate all Parquet files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
ROOT = Path("../")
CORR_DIR = ROOT / "data" / "correlations"


# ── Load files ────────────────────────────────────────────────────────────────
def load_correlations(method: str, metric: str) -> pd.DataFrame:
    path = CORR_DIR / f"correlations_{method}_{metric}.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"Run: uv run python scripts/precompute_correlations.py --method {method} --metric {metric}"
        )
    return pd.read_parquet(path)


stat = load_correlations("statistical", "perf_score")
print(f"statistical / perf_score: {stat.shape}")
stat.head(4)

## 1. Dataset Overview

In [ ]:
print("Characteristics:", sorted(stat["user_characteristic"].unique()))
print("Attributes:     ", sorted(stat["creative_attribute"].unique()))
print()

segment_counts = (
    stat.groupby("user_characteristic")["user_characteristic_value"].nunique().rename("n_values")
)
print("Segments per characteristic:")
print(segment_counts.to_string())
print()

# NaN rate per attribute (constant features in narrow segments)
nan_rate = (
    stat.groupby("creative_attribute")["correlation"].apply(lambda s: s.isna().mean()).round(3)
)
if nan_rate.gt(0).any():
    print("NaN correlation rate by attribute (constant-feature segments):")
    print(nan_rate[nan_rate > 0].sort_values(ascending=False).to_string())

## 2. Global Top Correlations with `perf_score`

Average absolute Pearson *r* across all segments to identify the most consistently impactful attributes.

In [ ]:
# Numeric/binary attributes only (level is NaN → attribute-level row)
numeric_rows = stat[stat["creative_attribute_level"].isna()].copy()

mean_abs = (
    numeric_rows.groupby("creative_attribute")["correlation"]
    .agg(mean_r="mean", mean_abs_r=lambda s: s.abs().mean(), std_r="std")
    .sort_values("mean_abs_r", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#27AE60" if v >= 0 else "#E74C3C" for v in mean_abs["mean_r"]]
bars = ax.barh(mean_abs.index, mean_abs["mean_r"], color=colors, edgecolor="white", alpha=0.85)
ax.errorbar(
    mean_abs["mean_r"],
    range(len(mean_abs)),
    xerr=mean_abs["std_r"],
    fmt="none",
    color="#333",
    capsize=3,
    linewidth=0.8,
)
ax.axvline(0, color="#333", linewidth=0.8)
ax.set_xlabel("Mean Pearson r across all segments (error bar = ±1 std)")
ax.set_title(
    "Global Attribute → perf_score Correlations\n(averaged across all segments)", fontweight="bold"
)
plt.tight_layout()
plt.show()

print(mean_abs.round(4).to_string())

## 3. Heatmap: Numeric Attributes × Age Segment

In [ ]:
def segment_heatmap(df: pd.DataFrame, characteristic: str, title: str, figsize=(12, 7)) -> None:
    """Pivot correlations for numeric/binary attributes into a characteristic × attribute heatmap."""
    sub = df[
        (df["user_characteristic"] == characteristic) & (df["creative_attribute_level"].isna())
    ].copy()
    pivot = sub.pivot(
        index="user_characteristic_value", columns="creative_attribute", values="correlation"
    )
    # Sort columns by mean absolute correlation
    pivot = pivot[pivot.abs().mean().sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        center=0,
        cmap="RdYlGn",
        linewidths=0.4,
        ax=ax,
        cbar_kws={"label": "Pearson r"},
        annot_kws={"size": 8},
    )
    ax.set_title(title, fontweight="bold", pad=12)
    ax.set_xlabel("Creative Attribute")
    ax.set_ylabel(characteristic)
    plt.xticks(rotation=35, ha="right", fontsize=8)
    plt.tight_layout()
    plt.show()


segment_heatmap(stat, "target_age_segment", "perf_score Correlations by Target Age Segment")

## 4. Heatmap: Numeric Attributes × Country

In [ ]:
segment_heatmap(stat, "country", "perf_score Correlations by Country", figsize=(13, 8))

## 5. Heatmap: Numeric Attributes × Vertical

In [ ]:
segment_heatmap(stat, "vertical", "perf_score Correlations by Vertical")

## 6. Categorical Attributes — Top & Bottom Levels

For each categorical attribute, rank its levels by mean Pearson *r* across all segments.

In [ ]:
cat_rows = stat[stat["creative_attribute_level"].notna()].copy()
cat_attrs = sorted(cat_rows["creative_attribute"].unique())

n_cols = 2
n_rows = int(np.ceil(len(cat_attrs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
axes = axes.flatten()

for i, attr in enumerate(cat_attrs):
    ax = axes[i]
    sub = cat_rows[cat_rows["creative_attribute"] == attr]
    level_mean = sub.groupby("creative_attribute_level")["correlation"].mean().sort_values()
    colors = ["#27AE60" if v >= 0 else "#E74C3C" for v in level_mean]
    level_mean.plot.barh(ax=ax, color=colors, edgecolor="white", alpha=0.85)
    ax.axvline(0, color="#333", linewidth=0.7)
    ax.set_title(attr, fontweight="bold", fontsize=9)
    ax.set_xlabel("Mean Pearson r", fontsize=8)
    ax.tick_params(axis="both", labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Categorical Attribute Levels — Mean Pearson r with perf_score\n(across all segments)",
    fontweight="bold",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()

## 7. Segment Variability — Which Attributes Behave Differently Across Segments?

High standard deviation across segment values → the attribute's effect on performance depends strongly on the audience.

In [ ]:
variability = (
    numeric_rows.groupby(["user_characteristic", "creative_attribute"])["correlation"]
    .std()
    .rename("std_r")
    .reset_index()
    .sort_values("std_r", ascending=False)
)

# Show top-5 most variable (attribute, characteristic) pairs
print("Most context-dependent attribute × characteristic pairs (high std across segment values):")
print(variability.head(10).to_string(index=False))

# Heatmap: characteristic × attribute std
var_pivot = variability.pivot(
    index="user_characteristic", columns="creative_attribute", values="std_r"
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    var_pivot,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.4,
    ax=ax,
    cbar_kws={"label": "std(Pearson r)"},
    annot_kws={"size": 7},
)
ax.set_title(
    "Correlation Variability across Segment Values\n(higher = attribute effect is audience-dependent)",
    fontweight="bold",
)
plt.xticks(rotation=35, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 8. Focus: `novelty_score` and `motion_score` Across Segments

Two of the strongest predictors globally — how consistent are they across audience segments?

In [ ]:
focus_attrs = ["novelty_score", "motion_score"]

fig, axes = plt.subplots(1, len(focus_attrs), figsize=(14, 5), sharey=False)

for ax, attr in zip(axes, focus_attrs):
    sub = numeric_rows[numeric_rows["creative_attribute"] == attr].copy()
    sub["label"] = sub["user_characteristic"] + "=" + sub["user_characteristic_value"]
    sub = sub.sort_values("correlation")

    colors = ["#27AE60" if v >= 0 else "#E74C3C" for v in sub["correlation"]]
    ax.barh(sub["label"], sub["correlation"], color=colors, edgecolor="white", alpha=0.85)
    ax.axvline(0, color="#333", linewidth=0.8)
    ax.set_title(f"{attr}\nvs perf_score", fontweight="bold")
    ax.set_xlabel("Pearson r")
    ax.tick_params(axis="y", labelsize=7)

plt.suptitle(
    "Per-Segment Correlation: key attributes vs perf_score", fontweight="bold", fontsize=12
)
plt.tight_layout()
plt.show()

## 9. Statistical vs RF Signed — Method Comparison

Load both methods for `perf_score` and compare rankings for numeric/binary attributes across all segments.

In [ ]:
try:
    rf = load_correlations("rf_signed", "perf_score")
    rf_available = True
except FileNotFoundError as e:
    print(e)
    rf_available = False

if rf_available:
    stat_num = stat[stat["creative_attribute_level"].isna()]
    rf_num = rf[rf["creative_attribute_level"].isna()]

    stat_mean = stat_num.groupby("creative_attribute")["correlation"].mean().rename("statistical")
    rf_mean = rf_num.groupby("creative_attribute")["correlation"].mean().rename("rf_signed")

    cmp = pd.concat([stat_mean, rf_mean], axis=1).dropna()
    # Normalise RF importances to [-1, 1] range for visual comparability
    rf_range = cmp["rf_signed"].abs().max()
    if rf_range > 0:
        cmp["rf_signed_norm"] = cmp["rf_signed"] / rf_range
    else:
        cmp["rf_signed_norm"] = cmp["rf_signed"]

    cmp = cmp.sort_values("statistical")

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(cmp))
    w = 0.35
    ax.barh(
        x - w / 2,
        cmp["statistical"],
        w,
        label="Pearson r (statistical)",
        color="#4C72B0",
        alpha=0.8,
    )
    ax.barh(
        x + w / 2,
        cmp["rf_signed_norm"],
        w,
        label="RF signed (normalised)",
        color="#DD8452",
        alpha=0.8,
    )
    ax.set_yticks(x)
    ax.set_yticklabels(cmp.index, fontsize=9)
    ax.axvline(0, color="#333", linewidth=0.8)
    ax.set_xlabel("Mean correlation across all segments")
    ax.set_title(
        "Statistical vs RF Signed — perf_score\nMean correlation per attribute across all segments",
        fontweight="bold",
    )
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Rank agreement
    cmp["rank_stat"] = cmp["statistical"].abs().rank(ascending=False).astype(int)
    cmp["rank_rf"] = cmp["rf_signed"].abs().rank(ascending=False).astype(int)
    cmp["rank_diff"] = (cmp["rank_stat"] - cmp["rank_rf"]).abs()
    print("Rank agreement (lower rank_diff = methods agree):")
    print(
        cmp[["rank_stat", "rank_rf", "rank_diff"]]
        .sort_values("rank_diff", ascending=False)
        .to_string()
    )

## 10. Statistical Significance Filter

Restrict to p < 0.05 and show how many attribute × segment pairs remain reliable.

In [ ]:
sig = stat[stat["creative_attribute_level"].isna()].copy()
sig["significant"] = sig["p_value"] < 0.05

sig_rate = (
    sig.groupby("creative_attribute")["significant"]
    .mean()
    .sort_values(ascending=False)
    .rename("pct_significant_segments")
)

fig, ax = plt.subplots(figsize=(9, 5))
sig_rate.plot.barh(ax=ax, color="#5B8DB8", edgecolor="white", alpha=0.85)
ax.axvline(0.5, color="#E74C3C", linewidth=1, linestyle="--", label="50% threshold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title(
    "% of Segments Where Correlation is Statistically Significant (p < 0.05)", fontweight="bold"
)
ax.set_xlabel("Fraction of segments with p < 0.05")
ax.legend()
plt.tight_layout()
plt.show()

print("\nAttributes significant in > 50% of segments:")
print(sig_rate[sig_rate > 0.5].round(3).to_string())

## 11. Summary Table — Strongest Actionable Signals

Combine magnitude, direction, and significance rate into a single ranked view.

In [ ]:
summary = (
    numeric_rows.groupby("creative_attribute")
    .agg(
        mean_r=("correlation", "mean"),
        mean_abs_r=("correlation", lambda s: s.abs().mean()),
        std_r=("correlation", "std"),
        pct_positive=("correlation", lambda s: (s > 0).mean()),
        pct_significant=("p_value", lambda s: (s < 0.05).mean()),
        n_segments=("correlation", "count"),
    )
    .sort_values("mean_abs_r", ascending=False)
)

summary["direction"] = summary["mean_r"].apply(
    lambda x: "positive" if x > 0.02 else ("negative" if x < -0.02 else "neutral")
)
summary["reliability"] = summary["pct_significant"].apply(
    lambda x: "high" if x > 0.7 else ("moderate" if x > 0.4 else "low")
)

display_cols = [
    "mean_r",
    "mean_abs_r",
    "std_r",
    "pct_positive",
    "pct_significant",
    "direction",
    "reliability",
]
print(summary[display_cols].round(3).to_string())